<a href="https://colab.research.google.com/github/TT0503/Thinktech/blob/TT0503-patch-1/Chilid_Mind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
eid# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


100%|██████████| 178M/178M [00:01<00:00, 139MB/s]

Extracting files...


100%|██████████| 226/226 [00:00<00:00, 433kB/s]

Extracting files...


100%|██████████| 245M/245M [00:03<00:00, 80.3MB/s]

Extracting files...


Data source import complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf

# Check available devices
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
TR = pd.read_csv('/content/drive/MyDrive/train.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(TR.shape)

NameError: name 'TR' is not defined

In [ ]:
!pip install --upgrade pip
!pip install polars


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.utils import Sequence, to_categorical, pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, add, MaxPooling1D, Dropout,
    Bidirectional, LSTM, GlobalAveragePooling1D, Dense, Multiply, Reshape,
    Lambda, Concatenate, GRU, GaussianNoise
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import tensorflow as tf
import polars as pl
from sklearn.model_selection import StratifiedGroupKFold
from scipy.spatial.transform import Rotation as R

In [ ]:
TS=pd.read_csv('/content/drive/MyDrive/test[1].csv')


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
TR = pd.read_csv('/content/drive/MyDrive/train.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
num_columns = TR.shape[1]
print(num_columns)

341


In [ ]:
# 1. Print column headings
print("Column headings:")
print(TR.columns.tolist())

# # 2. Print first 10 rows
# print("\nFirst 10 rows:")
# print(TR.head(10))

Column headings:
['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8', 'tof_1_v9', 'tof_1_v10', 'tof_1_v11', 'tof_1_v12', 'tof_1_v13', 'tof_1_v14', 'tof_1_v15', 'tof_1_v16', 'tof_1_v17', 'tof_1_v18', 'tof_1_v19', 'tof_1_v20', 'tof_1_v21', 'tof_1_v22', 'tof_1_v23', 'tof_1_v24', 'tof_1_v25', 'tof_1_v26', 'tof_1_v27', 'tof_1_v28', 'tof_1_v29', 'tof_1_v30', 'tof_1_v31', 'tof_1_v32', 'tof_1_v33', 'tof_1_v34', 'tof_1_v35', 'tof_1_v36', 'tof_1_v37', 'tof_1_v38', 'tof_1_v39', 'tof_1_v40', 'tof_1_v41', 'tof_1_v42', 'tof_1_v43', 'tof_1_v44', 'tof_1_v45', 'tof_1_v46', 'tof_1_v47', 'tof_1_v48', 'tof_1_v49', 'tof_1_v50', 'tof_1_v51', 'tof_1_v52', 'tof_1_v53', 'tof_1_v54', 'tof_1_v55', 'tof_1_v56', 'tof_1_v57', 'tof_1_v58

In [ ]:
print(TR.shape[0])


574945


In [ ]:
groups = TR.groupby('gesture')
gesture_column = TR['gesture']
print(gesture_column)
print(gesture_column[10:])
import pandas as pd
import os
change_mask = TR['gesture'] != TR['gesture'].shift()

# Get the rows where the gesture changes
gesture_transitions = gesture_column[change_mask]
gesture_transitions_df = gesture_transitions.reset_index()
gesture_transitions_df.columns = ['original_index', 'gesture']

# Save in current Kaggle working directory
gesture_transitions_df.to_excel("gesture_transitions.xlsx", index=False)

print("File saved in the current Kaggle directory.")

print(gesture_transitions[:])
y= gesture_transitions_df.to_numpy()
groups = TR.groupby('gesture').apply(lambda g: g.drop(columns=['behavior', 'phase', 'gesture', 'orientation', 'row_id', 'sequence_type'],))
print(type(y))
print(np.array(y).shape)
print(y[:10])

0         Cheek - pinch skin
1         Cheek - pinch skin
2         Cheek - pinch skin
3         Cheek - pinch skin
4         Cheek - pinch skin
                 ...        
574940     Write name on leg
574941     Write name on leg
574942     Write name on leg
574943     Write name on leg
574944     Write name on leg
Name: gesture, Length: 574945, dtype: object
10        Cheek - pinch skin
11        Cheek - pinch skin
12        Cheek - pinch skin
13        Cheek - pinch skin
14        Cheek - pinch skin
                 ...        
574940     Write name on leg
574941     Write name on leg
574942     Write name on leg
574943     Write name on leg
574944     Write name on leg
Name: gesture, Length: 574935, dtype: object
File saved in the current Kaggle directory.
0               Cheek - pinch skin
57        Forehead - pull hairline
125             Cheek - pinch skin
178              Write name on leg
239       Forehead - pull hairline
                    ...           
574606            

In [ ]:
groups = TR.groupby('gesture')
gesture_column = TR['gesture']
print(gesture_column)
print(gesture_column[10:])


0         Cheek - pinch skin
1         Cheek - pinch skin
2         Cheek - pinch skin
3         Cheek - pinch skin
4         Cheek - pinch skin
                 ...        
574940     Write name on leg
574941     Write name on leg
574942     Write name on leg
574943     Write name on leg
574944     Write name on leg
Name: gesture, Length: 574945, dtype: object
10        Cheek - pinch skin
11        Cheek - pinch skin
12        Cheek - pinch skin
13        Cheek - pinch skin
14        Cheek - pinch skin
                 ...        
574940     Write name on leg
574941     Write name on leg
574942     Write name on leg
574943     Write name on leg
574944     Write name on leg
Name: gesture, Length: 574935, dtype: object


In [ ]:
!pip install openpyxl

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(x, y, test_size=0.2)
y_train_1 = y_train[:, 0].reshape(-1, 1)
y_train_2 = y_train[:, 1].reshape(-1, 1)
y_train_3= y_train[:, 2].reshape(-1, 1)
y_val_1 = y_val[:, 0].reshape(-1, 1)
y_val_2 = y_val[:, 1].reshape(-1, 1)
y_val_3 = y_val[:, 2].reshape(-1, 1)


NameError: name 'x' is not defined

In [ ]:
classes = np.unique(np.concatenate((y_train, y_val), axis=0))
num_classes = len(np.unique(y_train))
# Count the occurrences of each category
unique_categories, counts = np.unique(y_val, return_counts=True)

# Create a dictionary to store the counts for each category
category_counts = dict(zip(unique_categories, counts))

# Print the counts for each category
for category, count in category_counts.items():
    print(f'{category}: {count}')


NameError: name 'y_train' is not defined

In [ ]:
import pandas as pd

# Columns you want to process
motion_sensors = [
    'acc_x', 'acc_y', 'acc_z',
    'rot_w', 'rot_x', 'rot_y', 'rot_z'
]
thermal_sensors = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_sensors = [col for col in TR.columns if col.startswith('tof_')]
sensor_columns = motion_sensors + thermal_sensors + tof_sensors
# sensor_columns = motion_sensors
# Store blocks for each column
blocks = {col: [] for col in sensor_columns}
current_blocks = {col: [] for col in sensor_columns}

# Loop through TR to collect blocks per gesture segment
for i in range(1, len(TR)):
    for col in sensor_columns:
        current_blocks[col].append(TR[col].iloc[i - 1])

    if TR['gesture'].iloc[i] != TR['gesture'].iloc[i - 1]:
        for col in sensor_columns:
            block_df = pd.DataFrame(current_blocks[col]).transpose()
            blocks[col].append(block_df)
            current_blocks[col] = []

# Handle the final gesture segment
for col in sensor_columns:
    current_blocks[col].append(TR[col].iloc[-1])
    block_df = pd.DataFrame(current_blocks[col]).transpose()
    blocks[col].append(block_df)

# Combine all blocks for each column into final DataFrames
final_dfs = {}
for col in sensor_columns:
    final_dfs[col] = pd.concat(blocks[col], axis=0, ignore_index=True).transpose()

# Example: access final_df for acc_x
print("acc_x shape:", final_dfs['acc_x'].shape)
print(final_dfs['acc_x'].head())
tensor_dfs = {}

for col, df in final_dfs.items():
    # Ensure all values are numeric and clean
    df_clean = df.astype('float32')  # Ensure dtype is compatible
    tensor_dfs[col] = tf.convert_to_tensor(df_clean.values, dtype=tf.float32)

KeyboardInterrupt: 

In [ ]:
import os

# Replace with your target directory path
directory_path = '/content/drive/MyDrive/sensor_corrected shape'

# Count CSV files
csv_files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]
print(f"Number of .csv files: {len(csv_files)}")

Number of .csv files: 332


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

file_path = "/content/drive/MyDrive/sensor_corrected shape"
sensor_files = []

# Find all .csv files
for file in os.listdir(file_path):
    if file.endswith(".csv"):
        sensor_files.append(file)

sensor_data = []


for csv_file in sensor_files:
    df = pd.read_csv(os.path.join(file_path, csv_file))


    values = df.values.astype(np.float32)
    if values.shape == (700, 7570):
        values = values.T

    if values.shape != (7570, 700):
        raise ValueError(f"File {csv_file} has incorrect shape: {values.shape}")
    sensor_data.append(values)

# Stack into a final tensor: shape (7570, 700, num_sensors)
final_tensor = tf.stack(sensor_data, axis=-1)  # (7570, 700, num_csv_files)

print("Final tensor shape:", final_tensor.shape)


Final tensor shape: (7570, 700, 332)


In [ ]:
x=final_tensor

In [ ]:
x= df_clean
print (x.shape)
print(x[:10])

NameError: name 'df_clean' is not defined

In [ ]:
y = y[:, 1]
print(y[:10])

['Cheek - pinch skin' 'Forehead - pull hairline' 'Cheek - pinch skin'
 'Write name on leg' 'Forehead - pull hairline'
 'Feel around in tray and pull out an object' 'Neck - scratch'
 'Neck - pinch skin' 'Forehead - pull hairline' 'Eyelash - pull hair']


In [ ]:
print(y.shape)

(7570,)


In [ ]:
from sklearn.model_selection import train_test_split

# If x and y are TensorFlow tensors, convert them to NumPy arrays
x_np = x.numpy() if isinstance(x, tf.Tensor) else x
y_np = y.numpy() if isinstance(y, tf.Tensor) else y

# Now split using sklearn
X_train, X_val, y_train, y_val = train_test_split(x_np, y_np, test_size=0.2, random_state=42)


In [ ]:
print (y.shape)

print(x.shape)

(7570,)
(7570, 700, 332)


In [ ]:
from sklearn.preprocessing import LabelEncoder
# Initialize encoder
label_encoder = LabelEncoder()

# Fit on all labels and transform
all_labels = np.concatenate((y_train, y_val), axis=0)
label_encoder.fit(all_labels)

# Transform y_train and y_val to integer labels
y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)

# You can get the class names like this:
classes = label_encoder.classes_
num_classes = len(classes)

# Count and print
unique_categories, counts = np.unique(y_val, return_counts=True)
category_counts = dict(zip(unique_categories, counts))

for category, count in category_counts.items():
    class_name = classes[category]
    print(f'{category} ({class_name}): {count}')


0 (Above ear - pull hair): 113
1 (Cheek - pinch skin): 135
2 (Drink from bottle/cup): 33
3 (Eyebrow - pull hair): 113
4 (Eyelash - pull hair): 120
5 (Feel around in tray and pull out an object): 30
6 (Forehead - pull hairline): 102
7 (Forehead - scratch): 116
8 (Glasses on/off): 30
9 (Neck - pinch skin): 108
10 (Neck - scratch): 121
11 (Pinch knee/leg skin): 37
12 (Pull air toward your face): 90
13 (Scratch knee/leg skin): 31
14 (Text on phone): 122
15 (Wave hello): 89
16 (Write name in air): 91
17 (Write name on leg): 33


In [ ]:

classes = np.unique(np.concatenate((y_train, y_val), axis=0))
num_classes = len(np.unique(y_train))
# Count the occurrences of each category
unique_categories, counts = np.unique(y_val, return_counts=True)

# Create a dictionary to store the counts for each category
category_counts = dict(zip(unique_categories, counts))

# Print the counts for each category
for category, count in category_counts.items():
    print(f'{category}: {count}')


0: 113
1: 135
2: 33
3: 113
4: 120
5: 30
6: 102
7: 116
8: 30
9: 108
10: 121
11: 37
12: 90
13: 31
14: 122
15: 89
16: 91
17: 33


In [ ]:
print(X_val.shape)

(1514, 700, 332)


In [ ]:


def build_deep_cnn1d_lstm_attention(input_shape, num_classes):
    input_layer = layers.Input(shape=input_shape)  # (timesteps, features)

    # ===== Conv1D Branch =====
    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)  # Dropout in CNN branch too

    # ===== BiLSTM + Attention Branch =====
    lstm = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3))(input_layer)
    lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.3))(lstm)
    attn_output = deep_attention_block(lstm)

    # ===== Feature Fusion =====
    merged = layers.Concatenate()([x, attn_output])

    # ===== Classification Head =====
    dense = layers.Dense(256, activation='relu')(merged)
    dense = layers.Dropout(0.5)(dense)
    dense = layers.Dense(128, activation='relu')(dense)
    dense = layers.Dropout(0.5)(dense)

    output = layers.Dense(num_classes, activation='softmax')(dense)

    model = models.Model(inputs=input_layer, outputs=output)
    return model

In [ ]:
from tensorflow.keras import layers, models

def build_deep_cnn1d_lstm(input_shape, num_classes):
    input_layer = layers.Input(shape=input_shape)  # (timesteps, features)

    # ===== Deep Conv1D Branch =====
    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)

    # ===== Deep BiLSTM Branch =====
    lstm = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3))(input_layer)
    lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.3))(lstm)
    lstm = layers.Bidirectional(layers.LSTM(32, return_sequences=True, dropout=0.3))(lstm)  # Added depth
    lstm = layers.GlobalAveragePooling1D()(lstm)
    lstm = layers.Dropout(0.5)(lstm)

    # ===== Feature Fusion =====
    merged = layers.Concatenate()([x, lstm])

    # ===== Deeper Classification Head =====
    dense = layers.Dense(512, activation='relu')(merged)
    dense = layers.Dropout(0.5)(dense)
    dense = layers.Dense(256, activation='relu')(dense)
    dense = layers.Dropout(0.5)(dense)
    dense = layers.Dense(128, activation='relu')(dense)
    dense = layers.Dropout(0.5)(dense)

    output = layers.Dense(num_classes, activation='softmax')(dense)

    model = models.Model(inputs=input_layer, outputs=output)
    return model

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Initialize encoder
label_encoder = LabelEncoder()

# Fit on all labels and transform
all_labels = np.concatenate((y_train, y_val), axis=0)
label_encoder.fit(all_labels)

# Transform y_train and y_val to integer labels
y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)

# You can get the class names like this:
classes = label_encoder.classes_
num_classes = len(classes)

# Count and print
unique_categories, counts = np.unique(y_val, return_counts=True)
category_counts = dict(zip(unique_categories, counts))

In [ ]:
def replace_nan_with_zero(arr):
    arr = np.array(arr)  # ensure numpy array
    arr = np.nan_to_num(arr, nan=0.0)
    return arr
X_val = replace_nan_with_zero(X_val)
X_train = replace_nan_with_zero(X_train)

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import models

In [ ]:
input_shape = (700, 332)  # For 1D input: time_steps × features
num_classes = len(np.unique(y_train))  # after label encoding

model = build_deep_cnn1d_lstm(input_shape, num_classes)
model.compile(optimizer='Adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Fit the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=500,
    batch_size=128,
    callbacks=[
        #tf.keras.callbacks.EarlyStopping(patience=50, restore_best_weights=True)
    ]
)
model.summary()

Epoch 1/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 29s 316ms/step - accuracy: 0.0727 - loss: 2.9142 - val_accuracy: 0.1083 - val_loss: 2.8224
Epoch 2/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.0873 - loss: 2.7980 - val_accuracy: 0.0779 - val_loss: 2.7178
Epoch 3/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 240ms/step - accuracy: 0.1563 - loss: 2.5917 - val_accuracy: 0.1995 - val_loss: 2.3526
Epoch 4/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 242ms/step - accuracy: 0.2097 - loss: 2.3313 - val_accuracy: 0.1354 - val_loss: 3.0553
Epoch 5/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.2327 - loss: 2.1695 - val_accuracy: 0.1962 - val_loss: 2.5869
Epoch 6/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 243ms/step - accuracy: 0.2670 - loss: 2.0487 - val_accuracy: 0.1651 - val_loss: 3.4186
Epoch 7/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.2707 - loss: 1.9726 - val_accuracy: 0.1618 - val_loss: 3.0806
Epoch 8/500
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.2875 - loss: 1.9139 - 

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 700, 332)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 700, 64)   │    106,304 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 700, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 700, 64)   │     20,544 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 350, 64)   │          0 │ conv1d_1[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 350, 64)   │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 350, 128)  │     41,088 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 350, 128)  │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 350, 128)  │     82,048 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 175, 128)  │          0 │ conv1d_3[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 175, 128)  │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 175, 256)  │     98,560 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 700, 256)  │    472,064 │ input_layer[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 175, 256)  │      1,024 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 700, 128)  │    164,352 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 175, 256)  │    196,864 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 700, 64)   │     41,216 │ bidirectional_1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv1d_5[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ bidirectional_2[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,665,400 (17.80 MB)

 Trainable params: 1,554,834 (5.93 MB)

 Non-trainable params: 896 (3.50 KB)

 Optimizer params: 3,109,670 (11.86 MB)